# YOLO Custom Object Detection Training Pipeline
Jupyter Notebook ini dirancang untuk alur kerja training YOLO custom pada sistem lokal menggunakan **PyTorch**, **CUDA**, dan GPU laptop (misal: RTX 3050 6GB).

### Alur Kerja:
1. Verifikasi Environment & GPU CUDA
2. Konfigurasi Dataset `data.yaml`
3. Setup Hyperparameter & Optimasi VRAM (Batch size, AMP, Workers)
4. Eksekusi Training Model
5. Evaluasi Metrik & Visualisasi Hasil Training
6. Inferensi & Uji Coba Model (`best.pt`)

## 1. Verifikasi Environment & Ketersediaan GPU

In [2]:
import os
import torch
from ultralytics import YOLO

print(f"PyTorch Version : {torch.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available  : {cuda_available}")

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Device Name     : {gpu_name}")
    print(f"VRAM Total      : {total_mem:.2f} GB")
else:
    print("PERINGATAN: GPU tidak terdeteksi oleh PyTorch. Training akan berjalan lambat di CPU.")

PyTorch Version : 2.14.1+cu130
CUDA Available  : True
Device Name     : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM Total      : 5.72 GB


## 2. Konfigurasi `data.yaml`
Sesuaikan path dataset dan daftar nama kelas sesuai folder proyek lokal kamu.

In [3]:
import yaml
from pathlib import Path

# Tentukan direktori absolut atau relatif dataset Anda
dataset_root = Path("./my_dataset").resolve()

data_config = {
    "path": str(dataset_root),
    "train": "images/train",
    "val": "images/val",
    "test": "images/val",  # Opsional: gunakan split val jika tidak ada test terpisah
    "names": {
        0: "vehicle",
        1: "pedestrian",
        2: "traffic_sign"
    }
}

# Simpan ke data.yaml jika belum ada
yaml_path = dataset_root / "data.yaml"
dataset_root.mkdir(parents=True, exist_ok=True)

with open(yaml_path, "w") as f:
    yaml.dump(data_config, f, sort_keys=False)

print(f"Konfigurasi data.yaml tersimpan di: {yaml_path}")

Konfigurasi data.yaml tersimpan di: /home/darklighs/Documents/yolo-training/my_dataset/data.yaml


## 3. Inisialisasi Model Pretrained
Untuk VRAM 6GB, model `yolo11n.pt` (Nano) atau `yolo11s.pt` (Small) memberikan keseimbangan terbaik antara kecepatan, akurasi, dan konsumsi memori.

In [4]:
# Inisialisasi bobot awal (bobot akan diunduh otomatis pada eksekusi pertama)
model_name = "yolo11n.pt"
model = YOLO(model_name)
print(f"Model {model_name} berhasil dimuat!")

Model yolo11n.pt berhasil dimuat!


## 4. Eksekusi Training
Parameter disesuaikan untuk laptop GPU 6GB:
- `batch=8` (atau 16 jika memori mencukupi)
- `imgsz=640`
- `workers=4` (mencegah beban berlebih pada thread DataLoader)
- `amp=True` (Automatic Mixed Precision untuk efisiensi VRAM)

In [5]:
training_results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,         # 0 untuk GPU NVIDIA pertama
    workers=4,
    amp=True,
    project="runs/detect",
    name="custom_yolo_train",
    exist_ok=True
)

Ultralytics 8.4.187 🚀 Python-3.14.8 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 5857MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/darklighs/Documents/yolo-training/my_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=cu

RuntimeError: Dataset '/home/darklighs/Documents/yolo-training/my_dataset/data.yaml' error ❌ Dataset '/home/darklighs/Documents/yolo-training/my_dataset/data.yaml' images not found, missing path '/home/darklighs/Documents/yolo-training/my_dataset/images/val'
Note dataset download directory is '/home/darklighs/Documents/yolo-training/datasets'. You can update this in '/home/darklighs/.config/Ultralytics/settings.json'

## 5. Visualisasi Hasil Training & Metrik mAP

In [ ]:
import matplotlib.pyplot as plt
import cv2

result_dir = Path("runs/detect/custom_yolo_train")
results_png = result_dir / "results.png"

if results_png.exists():
    img = cv2.imread(str(results_png))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(16, 9))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Grafik Loss & Metrik mAP", fontsize=14)
    plt.show()
else:
    print(f"File hasil grafik belum ditemukan di: {results_png}")

## 6. Inferensi / Prediksi Menggunakan Bobot Terbaik (`best.pt`)

In [ ]:
best_weights = result_dir / "weights" / "best.pt"

if best_weights.exists():
    trained_model = YOLO(str(best_weights))
    print(f"Bobot terbaik dimuat dari: {best_weights}")
    
    # Ganti path ini ke gambar uji coba Anda
    test_img = dataset_root / "images" / "val"
    
    # Jalankan prediksi
    preds = trained_model.predict(
        source=str(test_img),
        save=True,
        conf=0.25,
        device=0
    )
    print("Hasil prediksi tersimpan di folder 'runs/detect/predict'.")
else:
    print(f"File bobot {best_weights} belum tersedia. Jalankan proses training terlebih dahulu.")